# 市場空白地帯発見システム - 基礎分析 (EDA)

このノートブックでは、取得した飲食店データの基礎的な探索分析を行う。

## 内容
1. データ読み込み・基本統計量の確認
2. ジャンル別の店舗数分布
3. エリア別の店舗密度マップ
4. 評価・口コミ数の分布
5. 機会スコアの分布可視化

In [1]:
from __future__ import annotations

import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import folium
from folium.plugins import HeatMap
from IPython.display import IFrame, display

from config.settings import PROCESSED_DATA_DIR, RAW_DATA_DIR, TARGET_GENRES

## 1. データ読み込み

In [2]:
TAG = "tokyo"

agg_path = PROCESSED_DATA_DIR / f"{TAG}_aggregated.csv"
raw_path = RAW_DATA_DIR / f"{TAG}_hotpepper.csv"

try:
    df_agg = pd.read_csv(agg_path)
except FileNotFoundError:
    print(f"skip: aggregated data not found: {agg_path}")
    df_agg = pd.DataFrame(
        columns=["mesh_code", "unified_genre", "restaurant_count", "avg_rating", "total_reviews", "lat", "lng"]
    )

try:
    df_raw = pd.read_csv(raw_path)
except FileNotFoundError:
    print(f"skip: raw data not found: {raw_path}")
    df_raw = pd.DataFrame(columns=["id", "name", "genre", "lat", "lng", "rating", "review_count"])

if not df_agg.empty:
    display(df_agg.describe())
else:
    print("skip: df_agg is empty")

,restaurant_count,avg_rating,total_reviews,lat,lng
count,2003.000000,0.0,2003.0,2003.000000,2003.000000
mean,9.230654,NaN,0.0,35.685959,139.731234
std,20.967787,NaN,0.0,0.028802,0.033638
min,1.000000,NaN,0.0,35.631041,139.669126
25%,1.000000,NaN,0.0,35.662707,139.703429
50%,3.000000,NaN,0.0,35.686990,139.731354
75%,8.000000,NaN,0.0,35.708600,139.760886
max,349.000000,NaN,0.0,35.738836,139.791052


## 2. ジャンル別店舗数分布

In [3]:
if df_agg.empty:
    print("skip: df_agg is empty")
else:
    genre_counts = (
        df_agg.groupby("unified_genre", as_index=False)["restaurant_count"]
        .sum()
        .sort_values("restaurant_count", ascending=False)
    )
    fig = px.bar(
        genre_counts,
        x="unified_genre",
        y="restaurant_count",
        title="ジャンル別店舗数",
        labels={"unified_genre": "ジャンル", "restaurant_count": "店舗数"},
    )
    fig.show()

## 3. エリア別店舗密度マップ

In [4]:
if df_agg.empty:
    print("skip: df_agg is empty")
else:
    heatmap_df = df_agg.dropna(subset=["lat", "lng", "restaurant_count"]).copy()
    m = folium.Map(location=[35.69, 139.70], zoom_start=13)
    HeatMap(
        data=heatmap_df[["lat", "lng", "restaurant_count"]].values.tolist(),
        radius=18,
        blur=14,
        max_zoom=16,
    ).add_to(m)
    output_path = "../data/processed/tokyo_heatmap.html"
    m.save(output_path)
    display(IFrame(src=output_path, width="100%", height=600))

## 4. 評価・口コミ数の分布

In [5]:
if df_raw.empty:
    print("skip: df_raw is empty")
else:
    rating_df = df_raw.dropna(subset=["rating"]).copy()
    review_df = df_raw.dropna(subset=["review_count"]).copy()
    scatter_df = df_raw.dropna(subset=["rating", "review_count"]).copy()

    fig = px.histogram(
        rating_df, x="rating", nbins=30,
        title="評価の分布", labels={"rating": "評価"},
    )
    fig.show()

    fig = px.histogram(
        review_df, x="review_count", nbins=30,
        title="口コミ数の分布", labels={"review_count": "口コミ数"},
    )
    fig.show()

    fig = px.scatter(
        scatter_df, x="rating", y="review_count",
        hover_data=["name", "genre"] if {"name", "genre"}.issubset(scatter_df.columns) else None,
        title="評価と口コミ数の関係",
        labels={"rating": "評価", "review_count": "口コミ数"},
    )
    fig.show()

## 5. 機会スコアの分布可視化

In [6]:
scored_path = PROCESSED_DATA_DIR / f"{TAG}_scored.csv"

try:
    df_scored = pd.read_csv(scored_path)
except FileNotFoundError:
    print(f"skip: scored data not found: {scored_path}")
else:
    if df_scored.empty or "opportunity_score" not in df_scored.columns:
        print("skip: df_scored is empty or opportunity_score column is missing")
    else:
        fig = px.histogram(
            df_scored.dropna(subset=["opportunity_score"]),
            x="opportunity_score", nbins=30,
            title="機会スコアの分布", labels={"opportunity_score": "機会スコア"},
        )
        fig.show()

        top10 = (
            df_scored.dropna(subset=["opportunity_score"])
            .sort_values("opportunity_score", ascending=False)
            .head(10)
        )
        fig = px.bar(
            top10, x="mesh_code", y="opportunity_score",
            hover_data=["reason"] if "reason" in top10.columns else None,
            title="機会スコア上位10メッシュ",
            labels={"mesh_code": "メッシュコード", "opportunity_score": "機会スコア"},
        )
        fig.show()